# DeepCFD GNN Training Notebook

Questo notebook addestra una Graph Neural Network per predire $(U_x, U_y, p)$ su griglia CFD con ostacolo, a partire dagli stessi dati usati nei notebook CNN/U-Net.

## Scelte guidate dalla letteratura
- MeshGraphNets (Pfaff et al., ICLR 2021): message passing su mesh/grafi con feature geometriche relative sugli archi.
- Pratica PINN in fluidodinamica (Chuang and Barba, 2022): usare una componente data-driven forte e inserire i vincoli fisici in modo graduale per stabilita' training.

In pratica qui usiamo:
1. Grafo 4-neighbor della griglia $(H, W)$, con edge feature relative $(\Delta x, \Delta y, ||\Delta||)$. 
2. Node feature = canali input + coordinate normalizzate.
3. Message passing residuale stile Encode-Process-Decode.
4. Loss principale data-driven su regione fluida, con opzione physics-informed.

In [17]:
import os
import copy
import time
import pickle
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from src.models import PhysicsInformedLoss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Device: {device} | Seed: {SEED}')

Device: cuda | Seed: 42


In [ ]:
x = pickle.load(open('dataset/dataX.pkl', 'rb'))
y = pickle.load(open('dataset/dataY.pkl', 'rb'))

x = torch.as_tensor(x, dtype=torch.float32)
y = torch.as_tensor(y, dtype=torch.float32)

N, C_in, H, W = x.shape
_, C_out, _, _ = y.shape
print(f'X: {x.shape} | Y: {y.shape}')

channel_weights = torch.sqrt((y ** 2).mean(dim=(0, 2, 3))).view(1, -1, 1, 1).to(device)
print('Channel RMS weights:', channel_weights.flatten().cpu().numpy())

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(N, generator=g)
x = x[perm]
y = y[perm]

n_train = int(0.70 * N)
n_val = int(0.15 * N)

train_x, train_y = x[:n_train], y[:n_train]
val_x, val_y = x[n_train:n_train+n_val], y[n_train:n_train+n_val]
test_x, test_y = x[n_train+n_val:], y[n_train+n_val:]

print(f'Train: {len(train_x)} | Val: {len(val_x)} | Test: {len(test_x)}')

BATCH_SIZE = 32
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(test_x, test_y), batch_size=BATCH_SIZE, shuffle=False)

os.makedirs('checkpoints', exist_ok=True)
torch.save({
    'test_x': test_x,
    'test_y': test_y,
    'val_x': val_x,
    'val_y': val_y,
    'channel_weights': channel_weights.cpu()
}, 'checkpoints/data_splits.pt')
print('Saved checkpoints/data_splits.pt')

X: torch.Size([981, 3, 172, 79]) | Y: torch.Size([981, 3, 172, 79])
Channel RMS weights: [0.11532923 0.01745924 0.01346998]
Train: 686 | Val: 147 | Test: 148
Saved checkpoints/data_splits.pt


In [19]:
def build_grid_graph(H, W, device='cpu'):
    edges_src = []
    edges_dst = []
    edge_attr = []

    def node_id(i, j):
        return i * W + j

    for i in range(H):
        for j in range(W):
            u = node_id(i, j)
            for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ni, nj = i + di, j + dj
                if 0 <= ni < H and 0 <= nj < W:
                    v = node_id(ni, nj)
                    edges_src.append(u)
                    edges_dst.append(v)
                    dx = di / max(H - 1, 1)
                    dy = dj / max(W - 1, 1)
                    dist = (dx * dx + dy * dy) ** 0.5
                    edge_attr.append([dx, dy, dist])

    edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long, device=device)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float32, device=device)

    coords = []
    for i in range(H):
        for j in range(W):
            coords.append([i / max(H - 1, 1), j / max(W - 1, 1)])
    coords = torch.tensor(coords, dtype=torch.float32, device=device)

    return edge_index, edge_attr, coords

edge_index, edge_attr, node_coords = build_grid_graph(H, W, device=device)
print('Nodes:', H * W, '| Directed edges:', edge_index.shape[1])

Nodes: 13588 | Directed edges: 53850


In [20]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, n_layers=2, dropout=0.0):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(n_layers - 1):
            layers += [nn.Linear(d, hidden_dim), nn.ReLU(inplace=True)]
            if dropout > 0:
                layers += [nn.Dropout(dropout)]
            d = hidden_dim
        layers += [nn.Linear(d, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class GraphBlock(nn.Module):
    def __init__(self, latent_dim, edge_feat_dim=3):
        super().__init__()
        self.edge_mlp = MLP(2 * latent_dim + edge_feat_dim, latent_dim, latent_dim, n_layers=2)
        self.node_mlp = MLP(2 * latent_dim, latent_dim, latent_dim, n_layers=2)

    def forward(self, h, edge_index, edge_attr):
        src, dst = edge_index
        m_in = torch.cat([h[src], h[dst], edge_attr], dim=-1)
        m = self.edge_mlp(m_in)

        agg = torch.zeros_like(h)
        agg.index_add_(0, dst, m)

        h_upd = self.node_mlp(torch.cat([h, agg], dim=-1))
        return h + h_upd

class GridGNN(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, latent_dim=128, n_message_passing=6):
        super().__init__()
        self.encoder = MLP(in_channels + 2, latent_dim, latent_dim, n_layers=2)
        self.processor = nn.ModuleList([GraphBlock(latent_dim, edge_feat_dim=3) for _ in range(n_message_passing)])
        self.decoder = MLP(latent_dim, latent_dim, out_channels, n_layers=2)

    def _forward_single(self, x_single, edge_index, edge_attr, node_coords):
        # x_single: [C_in, H, W]
        C_in, H, W = x_single.shape
        x_nodes = x_single.permute(1, 2, 0).reshape(H * W, C_in)
        h = self.encoder(torch.cat([x_nodes, node_coords], dim=-1))
        for block in self.processor:
            h = block(h, edge_index, edge_attr)
        y_nodes = self.decoder(h)
        y_grid = y_nodes.reshape(H, W, -1).permute(2, 0, 1)
        return y_grid

    def forward(self, x, edge_index, edge_attr, node_coords):
        # x: [B, C_in, H, W]
        ys = []
        for b in range(x.shape[0]):
            ys.append(self._forward_single(x[b], edge_index, edge_attr, node_coords))
        return torch.stack(ys, dim=0)

model = GridGNN(in_channels=C_in, out_channels=C_out, latent_dim=128, n_message_passing=6).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable params:', f'{n_params:,}')

Trainable params: 629,379


In [21]:
def data_loss_fn(y_hat, y, channel_weights, region_mask=None):
    l_ux = ((y_hat[:, 0] - y[:, 0]) ** 2).unsqueeze(1)
    l_uy = ((y_hat[:, 1] - y[:, 1]) ** 2).unsqueeze(1)
    l_p = torch.abs(y_hat[:, 2] - y[:, 2]).unsqueeze(1)
    per_channel = torch.cat([l_ux, l_uy, l_p], dim=1) / channel_weights

    if region_mask is not None:
        mask = region_mask.expand_as(per_channel)
        return (per_channel * mask).sum() / mask.sum().clamp_min(1.0)
    return per_channel.mean()

@torch.no_grad()
def masked_mse(y_hat, y, region_mask):
    sq = (y_hat - y) ** 2
    me = region_mask.expand_as(sq)
    return (sq * me).sum().item() / me.sum().clamp_min(1.0).item()

def gradnorm_lambda(l_data, l_phys, output, eps=1e-8, clamp_range=(1e-3, 10.0)):
    g_d = torch.autograd.grad(l_data, output, retain_graph=True, create_graph=False)[0]
    g_p = torch.autograd.grad(l_phys, output, retain_graph=True, create_graph=False)[0]
    lam = (g_d.abs().mean() / (g_p.abs().mean() + eps)).detach()
    return torch.clamp(lam, *clamp_range)

def physics_ramp(epoch, start=20, end=80):
    if epoch < start:
        return 0.0
    if epoch > end:
        return 1.0
    return (epoch - start) / (end - start)

In [22]:
def train_one_epoch(model, loader, optimizer, channel_weights, epoch, physics_fn=None, lambda_phys=1e-2, adaptive=True):
    model.train()
    ramp = physics_ramp(epoch) if physics_fn is not None else 0.0

    totals = {'loss': 0.0, 'loss_data': 0.0, 'loss_phys': 0.0, 'mse': 0.0, 'lam_eff': 0.0}
    n_total = 0

    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()

        y_hat = model(bx, edge_index, edge_attr, node_coords)
        fluid_mask = (bx[:, 1:2] > 0.5).float()

        l_data = data_loss_fn(y_hat, by, channel_weights, region_mask=fluid_mask)
        l_phys = torch.zeros((), device=device)
        lam_dyn = torch.ones((), device=device)

        if physics_fn is not None and ramp > 0:
            phys = physics_fn(y_hat, bx)
            l_phys = phys['total_physics']
            if adaptive:
                lam_dyn = gradnorm_lambda(l_data, l_phys, y_hat)

        lam_eff = lambda_phys * ramp * lam_dyn
        loss = l_data + lam_eff * l_phys
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        bsz = bx.size(0)
        n_total += bsz
        totals['loss'] += loss.item() * bsz
        totals['loss_data'] += l_data.item() * bsz
        totals['loss_phys'] += float(l_phys) * bsz
        totals['lam_eff'] += float(lam_eff) * bsz
        totals['mse'] += masked_mse(y_hat.detach(), by, fluid_mask) * bsz

    for k in totals:
        totals[k] /= max(n_total, 1)
    return totals

@torch.no_grad()
def evaluate(model, loader, channel_weights, physics_fn=None):
    model.eval()
    totals = {'loss': 0.0, 'loss_phys': 0.0, 'mse': 0.0}
    n_total = 0

    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        y_hat = model(bx, edge_index, edge_attr, node_coords)
        fluid_mask = (bx[:, 1:2] > 0.5).float()

        l_data = data_loss_fn(y_hat, by, channel_weights, region_mask=fluid_mask)
        l_phys = torch.zeros((), device=device)
        if physics_fn is not None:
            l_phys = physics_fn(y_hat, bx)['total_physics']

        bsz = bx.size(0)
        n_total += bsz
        totals['loss'] += l_data.item() * bsz
        totals['loss_phys'] += float(l_phys) * bsz
        totals['mse'] += masked_mse(y_hat, by, fluid_mask) * bsz

    for k in totals:
        totals[k] /= max(n_total, 1)
    return totals

In [23]:
def run_experiment(tag, use_physics=False, epochs=120, lr=1e-3, lambda_phys=1e-2):
    model = GridGNN(in_channels=C_in, out_channels=C_out, latent_dim=128, n_message_passing=6).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=12)

    physics_fn = None
    if use_physics:
        base_phys = PhysicsInformedLoss().to(device)
        phys_weights = {'continuity': 1.0, 'momentum': 0.1, 'boundary': 1.0}
        physics_fn = lambda y_hat, x_in: base_phys(y_hat, x_in, phys_weights)

    best_state = None
    best_val = float('inf')
    history = {'train': [], 'val': []}

    t0 = time.time()
    for ep in range(1, epochs + 1):
        tr = train_one_epoch(model, train_loader, opt, channel_weights, epoch=ep, physics_fn=physics_fn, lambda_phys=lambda_phys)
        va = evaluate(model, val_loader, channel_weights, physics_fn=physics_fn)
        sched.step(va['loss'])

        history['train'].append(tr)
        history['val'].append(va)

        if va['loss'] < best_val:
            best_val = va['loss']
            best_state = copy.deepcopy(model.state_dict())

        if ep == 1 or ep % 20 == 0:
            msg = f"[{tag}] ep={ep:03d} lr={opt.param_groups[0]['lr']:.1e} train={tr['loss']:.3e} val={va['loss']:.3e} val_mse={va['mse']:.3e}"
            if use_physics:
                msg += f" val_phys={va['loss_phys']:.3e}"
            print(msg)

    elapsed = time.time() - t0
    model.load_state_dict(best_state)

    return {
        'tag': tag,
        'model': model,
        'history': history,
        'best_val_loss': best_val,
        'time': elapsed,
        'use_physics': use_physics
    }

In [24]:
EPOCHS = 120
results = {}

results['gnn_data'] = run_experiment('GNN (Data only)', use_physics=False, epochs=EPOCHS, lr=1e-3)
results['gnn_phys'] = run_experiment('GNN (+Physics)', use_physics=True, epochs=EPOCHS, lr=1e-3, lambda_phys=1e-2)

print('Training completed.')
for k, r in results.items():
    print(f"{k}: best_val={r['best_val_loss']:.4e} | time={r['time']:.1f}s")

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 31.73 GiB of which 8.25 MiB is free. Process 1226180 has 1.18 GiB memory in use. Process 1226740 has 818.00 MiB memory in use. Including non-PyTorch memory, this process has 29.74 GiB memory in use. Of the allocated memory 28.60 GiB is allocated by PyTorch, and 779.55 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
def evaluate_test(result_obj):
    model = result_obj['model']
    metrics = evaluate(model, test_loader, channel_weights, physics_fn=None)
    return metrics

test_metrics = {k: evaluate_test(v) for k, v in results.items()}
for k, m in test_metrics.items():
    print(f"{k}: test_loss={m['loss']:.4e}, test_mse={m['mse']:.4e}")

In [ ]:
os.makedirs('checkpoints', exist_ok=True)

for key, r in results.items():
    ckpt_path = f'checkpoints/model_{key}.pt'
    torch.save({
        'key': key,
        'tag': r['tag'],
        'builder': 'gnn_grid',
        'use_physics': r['use_physics'],
        'state_dict': r['model'].state_dict(),
        'best_val_loss': r['best_val_loss'],
        'time': r['time'],
        'graph': {'H': H, 'W': W, 'edges': int(edge_index.shape[1])},
        'in_channels': C_in,
        'out_channels': C_out,
    }, ckpt_path)
    print('Saved', ckpt_path)

history_dump = {k: v['history'] for k, v in results.items()}
with open('checkpoints/training_history_gnn.pkl', 'wb') as f:
    pickle.dump(history_dump, f)
print('Saved checkpoints/training_history_gnn.pkl')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))

for key, color in [('gnn_data', '#d62728'), ('gnn_phys', '#1f77b4')]:
    tr = [e['mse'] for e in results[key]['history']['train']]
    va = [e['mse'] for e in results[key]['history']['val']]
    ax[0].plot(tr, color=color, linestyle='--', label=f'{key} train')
    ax[0].plot(va, color=color, linestyle='-', label=f'{key} val')

ax[0].set_yscale('log')
ax[0].set_title('MSE curves')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Masked MSE')
ax[0].grid(True, alpha=0.3)
ax[0].legend(fontsize=8)

labels = list(test_metrics.keys())
vals = [test_metrics[k]['mse'] for k in labels]
ax[1].bar(labels, vals, color=['#d62728', '#1f77b4'])
ax[1].set_yscale('log')
ax[1].set_title('Test masked MSE')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/fig_gnn_training_summary.png', dpi=250)
plt.show()
print('Saved figures/fig_gnn_training_summary.png')